In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import require_widget
from lib.semantic_drift import (
    detect_semantic_invalidations,
    write_semantic_review_requests,
)
dbutils.widgets.text("catalog",         "")
dbutils.widgets.text("control_schema",  "uc_hygiene")
dbutils.widgets.text("lookback_days",   "7")
dbutils.widgets.text("dry_run",         "false")

catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
lookback_days  = int(dbutils.widgets.get("lookback_days") or "7")
dry_run        = dbutils.widgets.get("dry_run").lower() == "true"

print(f"Control: {catalog}.{control_schema}")
print(f"Lookback: {lookback_days}d")
print(f"Dry run: {dry_run}")
import time as _t; _task_start = _t.time()


In [0]:
# Detect schema drift events that invalidate semantic assets
# (certifications, glossary terms, metric view definitions)
invalidations = detect_semantic_invalidations(
    spark, catalog, control_schema, lookback_days
)

inv_count = invalidations.count()
print(f"Semantic invalidations: {inv_count}")
if inv_count > 0:
    invalidations.groupBy("invalidated_asset_type", "severity").count().show()


In [0]:
# Route review requests to owners (respects dry_run)
written = write_semantic_review_requests(
    spark, invalidations, catalog, control_schema, dry_run=dry_run
)


In [0]:
_duration = int(_t.time() - _task_start)
print(f"\n{'='*52}")
print(f"  SEMANTIC DRIFT REVIEW COMPLETE")
print(f"{'='*52}")
print(f"  Invalidations:    {inv_count}")
print(f"  Reviews routed:   {written}")
print(f"  Dry run:          {dry_run}")
print(f"  Duration:         {_duration}s")
print(f"{'='*52}")

from datetime import date
try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      DATE('{date.today()}'),
      'uc_steward_daily_governance',
      'p5_semantic_drift_review',
      'p5_observability',
      'success',
      {inv_count},
      {inv_count},
      {written if not dry_run else 0},
      {_duration},
      'lookback_days={lookback_days} dry_run={dry_run}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")
